<a href="https://colab.research.google.com/github/anjipandey16/AI-ML-DL_Projects/blob/main/Shakespare_NextWord_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib as mlt
import random
from datasets import load_dataset

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import tensorflow as tf

# Import necessary layers for building the neural network
from tensorflow.keras.layers import Dense, Dropout, LSTM, Activation
# Import the Sequential model API for creating a linear stack of layers
from tensorflow.keras.models import Sequential
# Import optimizers for updating the model's weights during training
from tensorflow.keras.optimizers import SGD, Adam
# Import callbacks for training behavior control
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from datetime import datetime

In [2]:
# 1. Download and load the raw WikiText-2 dataset
print("Downloading and loading WikiText-2...")
raw_datasets = load_dataset("wikitext", "wikitext-2-raw-v1")

# 2. Inspect the dataset splits
print("\nDataset Structure:")
print(raw_datasets)

# 3. Access individual splits
train_data = raw_datasets["train"]
val_data = raw_datasets["validation"]
test_data = raw_datasets["test"]

print(f"\nTraining samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Testing samples: {len(test_data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


Dataset Structure:
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Training samples: 36718
Validation samples: 3760
Testing samples: 4358


In [3]:
print("\n--- Sample Text Rows ---")
count = 0
while count < 3:
    random_idx = random.randint(0, len(train_data) - 1)
    text_sample = train_data[random_idx]["text"].strip()

    # WikiText contains empty rows or section headers wrapped in '='
    if text_sample and not text_sample.startswith("="):
        print(f"Sample {count + 1}: {text_sample}\n")
        count += 1


--- Sample Text Rows ---
Sample 1: Small firms dominate employment in the private sector . The Talisker Distillery , which produces a single malt whisky , is beside Loch Harport on the west coast of the island . Three other whiskies — Mac na Mara ( " son of the sea " ) , Tè Bheag nan Eilean ( " wee dram of the isles " ) and Poit Dhubh ( " black pot " ) — are produced by blender Pràban na Linne ( " smugglers den by the Sound of Sleat " ) , based at Eilean Iarmain . These are marketed using predominantly Gaelic @-@ language labels . There is also an established software presence on Skye , with Portree @-@ based Sitekit having expanded in recent years .

Sample 2: Bird One closes in on the American space capsule , and US forces prepare to launch a nuclear attack on the USSR . Meanwhile , the Japanese ninjas approach the base 's entrance , but are detected and fired upon . Bond manages to open the hatch , letting in the ninjas . During the ensuing battle , Bond fights his way to the contr

In [4]:
# 1. Clean and tokenize the raw text
def tokenize(text_data):
    tokens = []
    for row in text_data:
        text = row["text"].strip().lower()
        if text and not text.startswith("="):  # Skip headers and empty rows
            # Keep only words and simple punctuation
            words = re.findall(r"\b\w+\b|[.,!?;]", text)
            tokens.extend(words)
    return tokens

print("Tokenizing train and validation sets...")
train_tokens = tokenize(raw_datasets["train"])
val_tokens = tokenize(raw_datasets["validation"])

# 2. Build the vocabulary (keep words that appear at least 3 times)
MIN_FREQ = 3
word_counts = collections.Counter(train_tokens)
vocab = {word: idx + 2 for idx, (word, freq) in enumerate(word_counts.items()) if freq >= MIN_FREQ}

# Add special tokens
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1  # For unknown/rare words

# Reverse vocabulary for decoding later
id_to_word = {idx: word for word, idx in vocab.items()}
VOCAB_SIZE = len(vocab)
print(f"Vocabulary built. Total unique words: {VOCAB_SIZE}")

# 3. Convert word tokens to numeric IDs
def tokens_to_ids(tokens, vocab_dict):
    return [vocab_dict.get(token, vocab_dict["<UNK>"]) for token in tokens]

train_ids = tokens_to_ids(train_tokens, vocab)
val_ids = tokens_to_ids(val_tokens, vocab)

# 4. Create fixed-length training sequences
# Example: Use 30 words to predict the 31st word
SEQUENCE_LENGTH = 30

def create_sequences(numerical_ids, seq_len):
    inputs, targets = [], []
    for i in range(len(numerical_ids) - seq_len):
        inputs.append(numerical_ids[i : i + seq_len])
        targets.append(numerical_ids[i + seq_len])
    return torch.tensor(inputs), torch.tensor(targets)

print("Creating input and target sequences...")
X_train, y_train = create_sequences(train_ids, SEQUENCE_LENGTH)
X_val, y_val = create_sequences(val_ids, SEQUENCE_LENGTH)

# 5. Create PyTorch DataLoaders for training
BATCH_SIZE = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Shapes for model input:")
print(f"X_train shape: {X_train.shape} (Samples, Sequence Length)")
print(f"y_train shape: {y_train.shape} (Targets)")

Tokenizing train and validation sets...
Vocabulary built. Total unique words: 29316
Creating input and target sequences...
Shapes for model input:
X_train shape: torch.Size([1928513, 30]) (Samples, Sequence Length)
y_train shape: torch.Size([1928513]) (Targets)


In [5]:
model = Sequential()              # Create a Sequential model, which is a linear stack of layers
model.add(LSTM(80, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dense(units=1, activation='relu'))
model.summary()
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

early_stopping = EarlyStopping(monitor='val_loss',
                               patience=10,                 # Number of epochs to wait before stopping if validation loss does not improve
                               restore_best_weights=True    # Restore model weights from the epoch with the best performance
                               )
reduce_lr = ReduceLROnPlateau(monitor='val_loss',   # Monitor validation loss
                              factor=0.2,           # Reduce the learning rate by 20% when triggered
                              patience=5,           # Number of epochs to wait before reducing learning rate
                              min_lr=0.00001)
history = model.fit(X_train, y_train,       # Training data, Training labels
                    batch_size=32,          # Number of samples per gradient update
                    epochs=100,              # Maximum number of epochs to train
                    validation_split=0.2,   # Use 20% of the training data for validation
                    callbacks=[early_stopping, reduce_lr],  # Apply the defined callbacks during training
                    verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 80)         │        26,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        26,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,491 (205.04 KB)

 Trainable params: 52,491 (205.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
 4365/48213 ━━━━━━━━━━━━━━━━━━━━ 24:02 33ms/step - loss: 95114504.4701

KeyboardInterrupt: 

In [ ]:
# Define a function to calculate and print RMSE

def return_rmse(test,predicted):

    # Calculate the Root Mean Squared Error (RMSE) between test and predicted values

    rmse = math.sqrt(mean_squared_error(test, predicted))

    print("The root mean squared error is {}.".format(rmse))

# Create the testing data set
# Slice the scaled data from the end of the training set to the end of the dataset

test_data = scaled_data[training_data_len: , :]

# Extract the original dataset values for testing, starting from 60 time steps after the training data ends
y_test_org = dataset[training_data_len+60:, :]

# Prepare the input data for the model's predictions
x_test = []

# Loop through the test data, creating sequences of the last 60 time steps as input for each prediction
for i in range(60, len(test_data)):
    x_test.append(test_data[i-60:i, 0])

# Convert the list of sequences into a numpy array for model input
x_test = np.array(x_test)
# Reshape the input data to fit the LSTM model's expected input shape (samples, time steps, features)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1 ))

# Use the trained model to make predictions on the test data
predictions_scaled = model.predict(x_test)

# Inverse transform the predictions to get them back to the original scale
predictions_org = scaler.inverse_transform(predictions_scaled)


print(y_test_org.shape)
print(predictions_scaled.shape)

# Calculate and print the RMSE between the original test values and the model's predictions
return_rmse(y_test_org,predictions_org)

def plot_predictions(test,predicted):
    plt.plot(y_test_org, color='red',label='Real RIL Stock Price')
    plt.plot(predictions_org, color='blue',label='Predicted RIL Stock Price')
    plt.title('RIL Stock Price Prediction')
    plt.xlabel('Time')
    plt.ylabel('RIL Stock Price')
    plt.legend()
    plt.show()
plot_predictions(y_test_org,predictions_org)